# E02 · Map records, encounters and people into RDF

**Outcome:** Construct triples while keeping real-world identity, record identity and source identifiers distinct.

**Time:** about 40 minutes. Run cells in order. Edit the exercise cell after completing the walkthrough.

A CSV row is an information record. The hospital encounter it documents is an event unfolding in time. The patient is a person who may participate in several encounters. These deserve separate IRIs. Reusing the patient identifier as the row identifier would collapse repeated admissions. The course uses the source encounter_id and patient_nbr as local identifiers without claiming they are universal identities.

An RDF triple connects subject, predicate and object. Objects may be IRIs or literals. A typed integer supports numeric comparisons, while a string preserves the lexical diagnosis token. Never parse diagnosis codes as floating-point measurements: punctuation and leading zeros can carry code-system meaning. Human labels can change without changing the entity's IRI.

The adapter adds code-family types under an explicit local mapping policy. These are derived categorizations of code concepts, not new diagnoses of patients. The source CSV stays unchanged. A named graph can separate a transformation output from its source manifest, but graph names alone do not grant authorization or establish provenance. The pipeline must record who produced the graph and from which version.

In [1]:
from pathlib import Path
import sys, json
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ontology_lab").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Open this notebook from the extracted course folder.")
from ontology_lab import *
print("Course:", ROOT.name, "| source rows:", len(rows()))

Course: enterprise_ontology_tutorial | source rows: 60


## Build three triples by hand

In [2]:
r=rows()[0]
g=fresh_graph()
record=record_iri(r)
g.add((record,RDF.type,EX.EncounterRecord))
g.add((record,EX.encounterId,Literal(r['encounter_id'])))
g.add((record,EX.primaryCode,CODE[r['diag_1']]))
print(g.serialize(format='turtle'))

@prefix code: <https://example.org/health/icd9-source/> .
@prefix ex: <https://example.org/health/ontology/> .

<https://example.org/health/resource/record/2595612> a ex:EncounterRecord ;
    ex:encounterId "2595612" ;
    ex:primaryCode code:250.02 .




## Inspect the complete adapter output

In [3]:
g=build_asserted()
result=query(g,'SELECT ?id ?code ?days WHERE { ?r ex:encounterId ?id; ex:primaryCode ?code; ex:stayDays ?days } ORDER BY ?id LIMIT 5')
display(list(result))
assert len(members(g,EX.EncounterRecord))==60

[(rdflib.term.Literal('10195068'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/250.02'),
  rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('10323282'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/250.02'),
  rdflib.term.Literal('1', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('10555854'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/493'),
  rdflib.term.Literal('8', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('10590702'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/250.02'),
  rdflib.term.Literal('2', datatype=rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer'))),
 (rdflib.term.Literal('1157454'),
  rdflib.term.URIRef('https://example.org/health/icd9-source/428'),
  rdflib.term.Literal('7', datatype=rdflib.term.URIRef('http://www.w3

## Your turn

Construct the patient IRI for the first row using its source patient_nbr. Return a URIRef, not a literal.

Replace `answer = None` with your code. A skipped exercise is reported as incomplete; it is not a pass.

In [4]:
answer = None  # Write your solution here

In [5]:
learner_check(answer, lambda x: isinstance(x,URIRef) and str(x).endswith('/'+rows()[0]['patient_nbr']), 'Use RES and retain the patient identifier as text.')

Exercise not completed. Use RES and retain the patient identifier as text.
Out[0]: False


## Explain your model

What would break if a warehouse dimension surrogate key were treated as an enduring person identity?

Write a short answer below. Check the relevant chapter in the book before promoting a model change.

**My explanation:** _Write your explanation here._